In [18]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
import pandas as pd
from tqdm import tqdm
import tiktoken


In [19]:
# Carrega a chave da API do .env
load_dotenv()
client = OpenAI()

dataset_path = 'teams_messages_grouped_conversations.csv'



In [20]:
# Prompt com campo "email_solicitante"
prompt_base = f"""
Você é um analista inteligente de conversas de atendimento técnico.

Receberá abaixo uma transcrição de conversa entre um colaborador e o time de Help Desk. Sua tarefa é analisar e identificar **atendimentos reais de suporte**.

---

### Instruções:

1. **Identifique e separe atendimentos distintos**, baseando-se em:
   - Mudança de data;
   - Intervalos longos entre mensagens (mais de 60 minutos);
   - Novas mensagens que iniciam outro problema (ex: “bom dia, estou com outro problema…”).

2. **Ignore conversas que não representem uma solicitação de suporte técnico**, como:
   - Agradecimentos;
   - Confirmações de que o problema já foi resolvido;
   - Mensagens informativas sem solicitação de ação;
   - Mensagens genéricas ou elogios sem pedido de ajuda.

3. Considere como atendimento **válido** apenas quando a primeira mensagem da conversa expressar:
   - Um pedido de ajuda;
   - Um relato de problema;
   - Um sintoma ou erro observado (mesmo que sem pedir ajuda diretamente).

4. Para **cada atendimento válido identificado**, retorne os seguintes dados no formato JSON:

- `quem_solicitou_atendimento`: Nome completo da pessoa que solicitou suporte.
- `email_solicitante`: E-mail da pessoa que solicitou suporte.
- `quem_respondeu`: Nome(s) dos atendentes que se identificaram com algo como “Mateus aqui”, “Barbi aqui” etc.
- `data_solicitacao`: Data (AAAA-MM-DD) da primeira mensagem daquele atendimento.
- `hora_primeira_mensagem`: Hora da primeira mensagem (HH:MM:SS).
- `tempo_primeira_resposta`: Tempo entre a primeira mensagem do solicitante e a primeira resposta do Help Desk (em minutos e segundos).
- `tempo_total_atendimento`: Tempo entre a primeira e a última mensagem desse atendimento (também em minutos e segundos).
- `problema_relatado`: Resumo breve do problema relatado (com suas próprias palavras).
- `categoria`: Tente categorizar o problema relatado em algum tópico com poucas palavras,apenas para sumarizar tal atendimento.

5. Se nenhuma solicitação de atendimento técnico for encontrada na conversa, **retorne uma lista vazia**:

6. Caso o tempo_total_atendimento fique zerado, revise por gentileza e verifique se não houve erro de formatação, análise ou se a conversa não foi interrompida abruptamente.

```json
[]
"""

In [21]:
df = pd.read_csv(
    "teams_messages_grouped_conversations.csv",
    sep=",",  # ou ';', dependendo do seu CSV
    quotechar='"',  # ou '\'' dependendo de como foi salvo
    escapechar="\\",
    encoding="utf-8",
    engine="python",  # importante para lidar melhor com quebras de linha
)
# encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# resultados = [] 

# # Calcular total e média de tokens sem alterar o DataFrame
# tokens_total = 0
# tokens_lista = []


# # for conversa in df['ConversationHistory']:
# #     full_text = prompt_base + "\n\n" + str(conversa)
# #     print(full_text)
# #     num_tokens = len(encoding.encode(full_text))
# #     tokens_lista.append(num_tokens)
# #     tokens_total += num_tokens


    

# for _, row in tqdm(df.iterrows(), total=len(df)):
#     chat_id = row['ChatID']
#     conversa = row['ConversationHistory']

#     print(f"Processando ChatID: {chat_id} - {conversa}")
    
#     # Monta o prompt final para envio
#     prompt = f"{prompt_base}\n\n{conversa}\n\nRetorne a resposta em JSON conforme solicitado."

In [22]:
# Creating an array of json tasks

tasks = []

for index, row in df.iterrows():
    
    chat_id = row['ChatID']
    conversa = row['ConversationHistory']
    
    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4o-mini",
            "temperature": 0.1,
            "response_format": { 
                "type": "json_object"
            },
            "messages": [
                {
                    "role": "system",
                    "content": prompt_base
                },
                {
                    "role": "user",
                    "content": conversa
                }
            ],
        }
    }
    
    tasks.append(task)

In [23]:
#Creating the jsonl file
file_name = "tasks2.jsonl"

with open(file_name, 'w') as file:
    for task in tasks:
        file.write(json.dumps(task) + "\n")

In [24]:
#Uploading the file to OpenAI
batch_file = client.files.create(
    file=open(file_name, "rb"),
    purpose="batch",
)

print(f"Arquivo enviado com sucesso: {batch_file}")

Arquivo enviado com sucesso: FileObject(id='file-DjVJW2vwBTmgVn7ZR8LtcB', bytes=930519, created_at=1749474100, filename='tasks2.jsonl', object='file', purpose='batch', status='processed', expires_at=None, status_details=None)


In [25]:
#Creating the batch job
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h"
)

In [40]:
batch_job = client.batches.retrieve(batch_job.id)

In [41]:
result_file_id = batch_job.output_file_id
result = client.files.content(result_file_id).content

In [42]:
result_file_name = "results2.jsonl"

with open(result_file_name, 'wb') as file:
    file.write(result)

results = []
with open(result_file_name, 'r') as file:
    for line in file:
        # Parsing the JSON string into a dict and appending to the list of results
        json_object = json.loads(line.strip())
        results.append(json_object)

In [44]:
for res in results:
    result = res['response']['body']['choices'][0]['message']['content']
    print(result)
    
    

{
  "atendimentos": [
    {
      "quem_solicitou_atendimento": "Alberto Raymundo",
      "email_solicitante": "albertoraymundo@pecege.com",
      "quem_respondeu": "Help Desk Pecege",
      "data_solicitacao": "2025-03-11",
      "hora_primeira_mensagem": "16:49:39",
      "tempo_primeira_resposta": "00:01:44",
      "tempo_total_atendimento": "00:36:29",
      "problema_relatado": "Notebook não está imprimindo e apresenta erro ao tentar imprimir.",
      "categoria": "Impressão"
    },
    {
      "quem_solicitou_atendimento": "Alberto Raymundo",
      "email_solicitante": "albertoraymundo@pecege.com",
      "quem_respondeu": "Help Desk Pecege",
      "data_solicitacao": "2025-03-19",
      "hora_primeira_mensagem": "17:01:17",
      "tempo_primeira_resposta": "00:00:13",
      "tempo_total_atendimento": "00:27:53",
      "problema_relatado": "Impressão demora para sair e apresenta erro ao imprimir.",
      "categoria": "Impressão"
    }
  ]
}
{
  "atendimentos": [
    {
      "quem_